# FABLE Pyculator 2021 Scenario-Definition Patch

This notebook is a hands-on Phase 26 example for validating conservative edits to the 2021 FABLE-C `SCENARIOS definition` sheet.

No source workbook was mutated. A valid patch becomes generated-model input overrides keyed by workbook cell reference.

Expected local workbook artifact:

- `tmp/private-workbooks/2021_Open_FABLECalculator.xlsx`

Tracked example patch:

- `examples/scenario-definition-patches/fable_2021_diet_target_demo.yaml`

Optional generated-model artifact for the run cell:

- `tmp/generated-models/fable-2021/generated_fable_2021_model.py`

The generated model can be restored from:

- `examples/fable_2021/generated_fable_2021_model.py.xz`


In [ ]:
from pathlib import Path
import lzma
import shutil

import pandas as pd
from IPython.display import Markdown, display

from fable_pyculator import (
    DEFAULT_2021_GENERATED_MODEL_PATH,
    DEFAULT_2021_WORKBOOK_PATH,
    build_2021_notebook_spec,
    editable_scenario_definition_cells,
    load_scenario_definition_patch,
    output_table_frame,
    run_2021_notebook_loop,
    validate_scenario_definition_patch,
)


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for path in (current, *current.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "fable_pyculator").exists():
            return path
    raise RuntimeError("Could not find fable-pyculator repository root")


repo_root = find_repo_root(Path.cwd())
workbook_path = repo_root / DEFAULT_2021_WORKBOOK_PATH
generated_model_path = repo_root / DEFAULT_2021_GENERATED_MODEL_PATH
generated_model_archive_path = repo_root / "examples" / "fable_2021" / "generated_fable_2021_model.py.xz"
patch_path = repo_root / "examples" / "scenario-definition-patches" / "fable_2021_diet_target_demo.yaml"

if not generated_model_path.exists() and generated_model_archive_path.exists():
    generated_model_path.parent.mkdir(parents=True, exist_ok=True)
    with lzma.open(generated_model_archive_path, "rb") as source, generated_model_path.open("wb") as target:
        shutil.copyfileobj(source, target)

missing = [path for path in (workbook_path, patch_path) if not path.exists()]
ARTIFACTS_READY = not missing

display(Markdown(f"Repository root: `{repo_root}`"))
display(Markdown(f"Workbook path: `{workbook_path.relative_to(repo_root)}`"))
display(Markdown(f"Patch path: `{patch_path.relative_to(repo_root)}`"))
display(Markdown(f"Generated model path: `{generated_model_path.relative_to(repo_root)}`"))

if missing:
    display(Markdown("Missing required artifact(s):"))
    display([str(path.relative_to(repo_root)) for path in missing])
else:
    display(Markdown("Local workbook and patch are ready."))


## Discover Editable Cells

FABLE Pyculator derives the editable surface from workbook metadata. Phase 26 defaults to non-formula cells in columns tagged `DIRECT`.

In [ ]:
if ARTIFACTS_READY:
    spec = build_2021_notebook_spec(workbook_path)
    editable_cells = editable_scenario_definition_cells(spec)
    display(Markdown(f"Discovered `{len(editable_cells):,}` editable scenario-definition cells."))
    display(pd.DataFrame([cell.to_dict() for cell in editable_cells[:10]]))
else:
    spec = None
    editable_cells = ()
    display(Markdown("Restore the 2021 workbook before discovering editable cells."))


## Validate the Patch

The example patch uses a unique table/row/column selector. Validation resolves that selector to a workbook cell and produces generated-model input overrides. The workbook file is not edited.

In [ ]:
patch = load_scenario_definition_patch(patch_path)
display(patch.to_dict())

if spec is not None:
    patch_result = validate_scenario_definition_patch(spec, patch)
    display(Markdown(f"Validated `{len(patch_result.edits)}` edit(s)."))
    display(pd.DataFrame([
        {
            "cell_ref": edit["cell"]["cell_ref"],
            "table": edit["cell"]["table_label"],
            "row_label": edit["cell"]["row_label"],
            "column_label": edit["cell"]["column_label"],
            "original_value": edit["cell"]["original_value"],
            "patched_value": edit["value"],
            "role_tag": edit["cell"]["column_role_tag"],
        }
        for edit in patch_result.edits
    ]))
    display(Markdown("Generated-model input overrides:"))
    display(patch_result.inputs)
    display(Markdown("No source workbook was mutated."))
else:
    patch_result = None
    display(Markdown("Skipped patch validation because the workbook spec is unavailable."))


## Optional Generated-Model Run

The run is gated. Review the validated patch first, then set `RUN_MODEL = True` if the local 2021 workbook and generated model are available.

In [ ]:
RUN_MODEL = False

if RUN_MODEL:
    if not ARTIFACTS_READY:
        raise RuntimeError("Restore the 2021 workbook and patch before running the generated model.")
    if not generated_model_path.exists():
        raise RuntimeError(f"Generated model not found: {generated_model_path.relative_to(repo_root)}")

    result = run_2021_notebook_loop(
        {"gdp_scen": "SSP1"},
        workbook_path=workbook_path,
        generated_model_path=generated_model_path,
        scenario_definition_patch=patch,
        output_table_names=("ghg_resultsghg",),
        output_table_column_flavour_tags="OUTPUT-*",
        headline_series_names=("ghg_total_co2e",),
        include_figures=False,
    )
    display(Markdown("Generated model ran with the scenario-definition patch."))
    display({key: result.run.inputs[key] for key in patch_result.inputs})
    display(output_table_frame(result.run, "ghg_resultsghg", column_flavour_tags="OUTPUT-8").head())
    display(result.headline_frames["ghg_total_co2e"].head())
else:
    display(Markdown("Model run not started. Set `RUN_MODEL = True` after reviewing the patch validation output."))


## Useful Feedback to Send Back

If something fails, the most useful details are:

- the patch YAML you used;
- the validation cell output;
- whether the target cell was missing, read-only, formula-valued, duplicated, or ambiguous;
- the generated-model run cell output, if you enabled `RUN_MODEL`.
